# `Parallel and Conditional Chains`
---

In **LangChain**, chains let you connect multiple operations into a workflow. Two important patterns are:

1. **Parallel Chains** → execute multiple chains at the same time.
2. **Conditional Chains** → choose which chain to execute based on the input.

These are especially useful when building **LLM applications, RAG systems, and AI agents**.

---

# 1. What is a Chain?

A chain is simply a sequence of operations:

**Input → Prompt → LLM → Output**

For example:

```text
Question
   ↓
Prompt Template
   ↓
LLM
   ↓
Answer
```

In modern LangChain, these workflows are commonly built using **LCEL — LangChain Expression Language**.

Example:

```python
chain = prompt | llm | parser
```

Here:

* `prompt` formats the input
* `llm` generates the response
* `parser` converts the response into the desired format

---

# 2. Sequential Chain

Before understanding parallel and conditional chains, understand the normal sequential chain.

Suppose we want:

```text
Question
   ↓
Generate Answer
   ↓
Translate Answer
   ↓
Final Output
```

Code:

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini")

answer_prompt = ChatPromptTemplate.from_template(
    "Answer this question:\n{question}"
)

translate_prompt = ChatPromptTemplate.from_template(
    "Translate this answer into Hindi:\n{answer}"
)

answer_chain = answer_prompt | llm | StrOutputParser()

translate_chain = (
    translate_prompt
    | llm
    | StrOutputParser()
)

chain = (
    answer_chain
    | (lambda answer: {"answer": answer})
    | translate_chain
)
```

The important idea is:

```text
A → B → C
```

Each step waits for the previous step.

---

# 3. Parallel Chains

A **parallel chain** runs multiple operations independently using the **same input**.

For example, given a topic:

```text
             ┌──→ Generate Summary
Topic ───────┼──→ Generate Keywords
             └──→ Generate Questions
```

All three operations can execute independently.

This is useful when the outputs don't depend on each other.

## Example

Suppose the user gives:

```text
"Artificial Intelligence"
```

We want:

* Summary
* Key points
* Interview questions

We can run these three LLM calls in parallel.

### Code

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

llm = ChatOpenAI(model="gpt-4o-mini")

summary_prompt = ChatPromptTemplate.from_template(
    "Give a short summary of {topic}"
)

keypoints_prompt = ChatPromptTemplate.from_template(
    "Give 5 important key points about {topic}"
)

questions_prompt = ChatPromptTemplate.from_template(
    "Generate 5 interview questions about {topic}"
)

summary_chain = summary_prompt | llm | StrOutputParser()

keypoints_chain = keypoints_prompt | llm | StrOutputParser()

questions_chain = questions_prompt | llm | StrOutputParser()
```

Now create the parallel chain:

```python
parallel_chain = RunnableParallel(
    summary=summary_chain,
    keypoints=keypoints_chain,
    questions=questions_chain
)
```

Run it:

```python
result = parallel_chain.invoke({
    "topic": "Artificial Intelligence"
})

print(result)
```

The result will look approximately like:

```python
{
    "summary": "...",
    "keypoints": "...",
    "questions": "..."
}
```

---

# 4. How Parallel Chains Work

The important part is:

```python
RunnableParallel(
    summary=summary_chain,
    keypoints=keypoints_chain,
    questions=questions_chain
)
```

Conceptually:

```text
                  ┌───────────────┐
                  │ Summary Chain │
                  └───────┬───────┘
                          │
                          │
Input ────────────────────┼──────────────→ Output
                          │
                  ┌───────▼────────┐
                  │ Keypoint Chain │
                  └───────┬────────┘
                          │
                          │
                  ┌───────▼─────────┐
                  │ Question Chain  │
                  └─────────────────┘
```

All three receive the same input.

---

# 5. Why Use Parallel Chains?

Suppose you are building a document-analysis application.

You need:

```text
Document
   │
   ├── Summary
   ├── Sentiment
   ├── Keywords
   └── Topics
```

These operations don't necessarily depend on each other.

Sequential execution:

```text
Document
   ↓
Summary
   ↓
Sentiment
   ↓
Keywords
   ↓
Topics
```

This can be slower.

Parallel execution:

```text
             ┌── Summary
             │
Document ────┼── Sentiment
             │
             ├── Keywords
             │
             └── Topics
```

can reduce overall latency because independent work can happen concurrently.

---

# 6. Parallel Chain with a Common Input

Another common pattern is:

```python
parallel_chain = RunnableParallel(
    summary=summary_chain,
    keywords=keywords_chain
)
```

Input:

```python
{
    "topic": "Machine Learning"
}
```

Output:

```python
{
    "summary": "...",
    "keywords": "..."
}
```

Notice that both chains consume:

```python
topic
```

but produce different outputs.

---

# 7. Parallel Chain + Sequential Chain

Parallel and sequential chains can also be combined.

For example:

```text
                  ┌──→ Summary ───┐
Document ─────────┤               │
                  ├──→ Keywords ──┤
                  │               ↓
                  └──→ Topics ───→ Combine
```

Code:

```python
parallel_chain = RunnableParallel(
    summary=summary_chain,
    keywords=keywords_chain,
    topics=topics_chain
)
```

Then:

```python
final_chain = parallel_chain | final_prompt | llm
```

So the workflow becomes:

```text
Input
 ↓
Parallel Processing
 ↓
Combined Result
 ↓
LLM
 ↓
Final Answer
```

This is extremely useful in real-world LLM applications.

---

# 8. Conditional Chains

A **conditional chain** chooses a different workflow depending on the input.

Think of it as:

```text
                ┌── Condition A → Chain A
Input ──────────┤
                └── Condition B → Chain B
```

For example, suppose we have a customer-support chatbot.

If the user asks about:

```text
technical issue
```

→ use technical-support chain.

If the user asks about:

```text
billing
```

→ use billing-support chain.

```text
                    ┌── Technical Chain
                    │
User Question ──────┤
                    │
                    └── Billing Chain
```

---

# 9. RunnableBranch

LangChain provides `RunnableBranch` for this pattern.

Example:

```python
from langchain_core.runnables import RunnableBranch
```

Create two prompts:

```python
technical_prompt = ChatPromptTemplate.from_template(
    """
    You are a technical support specialist.

    Answer this question:
    {question}
    """
)

billing_prompt = ChatPromptTemplate.from_template(
    """
    You are a billing support specialist.

    Answer this question:
    {question}
    """
)
```

Create chains:

```python
technical_chain = technical_prompt | llm | StrOutputParser()

billing_chain = billing_prompt | llm | StrOutputParser()
```

Now create the conditional chain:

```python
conditional_chain = RunnableBranch(
    (
        lambda x: "error" in x["question"].lower(),
        technical_chain
    ),
    (
        lambda x: "payment" in x["question"].lower(),
        billing_chain
    ),
    technical_chain
)
```

The last argument is the **default branch**.

---

# 10. How RunnableBranch Works

Suppose input is:

```python
{
    "question": "My payment failed"
}
```

LangChain checks:

```python
"error" in question
```

False.

Then:

```python
"payment" in question
```

True.

Therefore:

```text
User Question
      ↓
Condition 1?
      ↓ No
Condition 2?
      ↓ Yes
Billing Chain
```

The billing chain executes.

---

# 11. Multiple Conditions

You can have several branches:

```python
conditional_chain = RunnableBranch(
    (
        lambda x: x["type"] == "technical",
        technical_chain
    ),
    (
        lambda x: x["type"] == "billing",
        billing_chain
    ),
    (
        lambda x: x["type"] == "sales",
        sales_chain
    ),
    general_chain
)
```

Conceptually:

```text
                    ┌── technical → Technical Chain
                    │
                    ├── billing ──→ Billing Chain
Input ── Condition ─┼── sales ────→ Sales Chain
                    │
                    └── otherwise → General Chain
```

Conditions are evaluated in order.

The first matching condition wins.

---

# 12. Parallel vs Conditional

The biggest difference is **execution logic**.

| Feature      | Parallel Chain                         | Conditional Chain           |
| ------------ | -------------------------------------- | --------------------------- |
| Purpose      | Run multiple chains                    | Select one chain            |
| Main class   | `RunnableParallel`                     | `RunnableBranch`            |
| Execution    | Multiple branches                      | Usually one matching branch |
| Input        | Usually shared                         | Used to evaluate conditions |
| Example      | Summary + keywords                     | Technical vs billing        |
| Main benefit | Lower latency / independent processing | Dynamic routing             |

Think:

### Parallel

```text
Input
 ↓
├── A
├── B
└── C
```

### Conditional

```text
Input
 ↓
Condition
 ↓
├── A
├── B
└── C
```

---

# 13. Real-World RAG Example

This becomes particularly important in **RAG systems**.

Imagine a chatbot that handles:

* General questions
* Company-document questions
* SQL questions

You could build:

```text
                         ┌── General LLM
                         │
User Question → Router ──┼── RAG Retriever → LLM
                         │
                         └── SQL Chain
```

The router determines what type of question the user asked.

For example:

```text
"What is machine learning?"
```

→ General LLM

```text
"What is our company's leave policy?"
```

→ RAG

```text
"How many customers registered last month?"
```

→ SQL chain

This is a **conditional routing architecture**.

---

# 14. Combining Parallel + Conditional Chains

You can combine both patterns.

For example:

```text
                         ┌── Summary
                         │
Document → Router ───────┼── Keywords
                         │
                         └── Questions
```

If the document is a certain type, choose a workflow, and inside that workflow perform multiple operations in parallel.

Another example:

```text
User Query
    ↓
Router
    ↓
┌──────────────┐
│              │
RAG          SQL
│              │
├── Search     ├── Query DB
├── Metadata   │
└── Rerank     │
│              │
└──────┬───────┘
       ↓
     Answer
```

This is closer to how production LLM applications are architected.

---

# 15. Important LCEL Concept

These components are **Runnables**.

For example:

```python
prompt
```

is a runnable.

```python
llm
```

is a runnable.

```python
parser
```

is a runnable.

And:

```python
RunnableParallel(...)
```

is also a runnable.

```python
RunnableBranch(...)
```

is also a runnable.

Therefore, you can compose them:

```python
chain = prompt | llm | parser
```

or:

```python
chain = RunnableParallel(...) | another_chain
```

or:

```python
chain = RunnableBranch(...) | parser
```

This composability is one of the core ideas behind **LCEL**.

---

# 16. Simple Mental Model

Remember these three patterns:

### Sequential

```text
A → B → C
```

**Do this, then this, then this.**

### Parallel

```text
      ┌→ A
Input ├→ B
      └→ C
```

**Do these independent tasks at the same time.**

### Conditional

```text
       ┌→ A
Input → Condition
       ├→ B
       └→ C
```

**Decide which task should execute.**

---

# 17. When to Use Which?

Use **Sequential Chains** when:

```text
Output of A is required by B.
```

Use **Parallel Chains** when:

```text
A, B and C are independent.
```

Use **Conditional Chains** when:

```text
The appropriate chain depends on the input.
```

In a production GenAI application, you will often use all three:

```text
                    ┌──→ Chain A ──┐
Input → Router ─────┼──→ Chain B ──┼──→ Final Chain
                    └──→ Chain C ──┘
```

or:

```text
Input
 ↓
Sequential
 ↓
Parallel
 ↓
Sequential
 ↓
Output
```

The key distinction is:

> **Parallel = execute multiple paths. Conditional = select a path.**

